# AMD Hackathon — Vulnerability-Aware Java LLM

Script 1: Download all 4 datasets to ./raw_data/

Datasets:
  1. barryallen16/java_vulnerability_dataset  (HuggingFace)
  2. JavaVFC                                  (Zenodo 13731781)
  3. CWE-Bench-Java                           (iris-sast GitHub)
  4. CVEFixes                                 (Kaggle - requires API key)

Usage:
  pip install huggingface_hub kaggle requests tqdm
  python 01_download_datasets.py

  For Kaggle (Dataset 4):
    - Create ~/.kaggle/kaggle.json  OR  set KAGGLE_USERNAME + KAGGLE_KEY env vars
    - kaggle.json format: {"username":"<user>","key":"<key>"}

In [ ]:
import os
import sys
import json
import shutil
import hashlib
import argparse
import urllib.request
import urllib.error
import ssl
from pathlib import Path

# ── optional heavy deps ───────────────────────────────────────────────────────
try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

RAW_DIR = Path("raw_data")

## Helpers

In [ ]:
def _ssl_ctx():
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    return ctx

In [ ]:
def _progress_hook(description: str):
    """Returns a reporthook for urllib that shows a simple progress bar."""
    seen = [0]

    def hook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        if total_size > 0:
            pct = min(100.0, downloaded * 100.0 / total_size)
            mb_done = downloaded / 1e6
            mb_total = total_size / 1e6
            print(
                f"\r  {description}: {pct:5.1f}%  {mb_done:.1f}/{mb_total:.1f} MB",
                end="",
                flush=True,
            )
        else:
            mb_done = downloaded / 1e6
            print(f"\r  {description}: {mb_done:.1f} MB downloaded", end="", flush=True)
        if block_num * block_size >= total_size and total_size > 0:
            print()

    return hook

In [ ]:
def safe_download(url: str, dest: Path, description: str = "", skip_if_exists: bool = True):
    """Download url → dest with optional progress, skipping if already exists."""
    if skip_if_exists and dest.exists() and dest.stat().st_size > 1024:
        print(f"  [SKIP] {dest.name} already exists.")
        return True

    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"  Downloading → {dest}")
    try:
        req = urllib.request.Request(
            url,
            headers={
                "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) python-downloader/1.0",
                "Accept": "*/*",
            },
        )
        with urllib.request.urlopen(req, timeout=300, context=_ssl_ctx()) as resp:
            total = int(resp.headers.get("Content-Length", 0))
            chunk = 65536
            downloaded = 0
            with open(dest, "wb") as f:
                while True:
                    data = resp.read(chunk)
                    if not data:
                        break
                    f.write(data)
                    downloaded += len(data)
                    if total:
                        pct = min(100.0, downloaded * 100.0 / total)
                        print(f"\r  {description}: {pct:.1f}%  ({downloaded/1e6:.1f}/{total/1e6:.1f} MB)", end="", flush=True)
        print(f"\r  {description}: Done — {downloaded/1e6:.1f} MB       ")
        return True
    except urllib.error.HTTPError as e:
        print(f"\n  [ERROR] HTTP {e.code}: {url}")
        return False
    except Exception as e:
        print(f"\n  [ERROR] {e}: {url}")
        return False

## Dataset 1 — barryallen16/java_vulnerability_dataset  (HuggingFace)

In [ ]:
def download_barryallen16():
    """
    Downloads the barryallen16/java_vulnerability_dataset from HuggingFace.
    Files: java_vul_dataset.csv, final_testcases.csv, finetuning_dataset.jsonl (v1–v4)
    """
    print("\n[1/4] barryallen16/java_vulnerability_dataset (HuggingFace)")
    out_dir = RAW_DIR / "barryallen16"
    out_dir.mkdir(parents=True, exist_ok=True)

    base = "https://huggingface.co/datasets/barryallen16/java_vulnerability_dataset/resolve/main/"
    files = [
        "java_vul_dataset.csv",
        "final_testcases.csv",
        "finetuning_dataset.jsonl",
        "finetuning_datasetv2.jsonl",
        "finetuning_dataset_v3.jsonl",
        "finetuning_dataset_v4.jsonl",
    ]

    for fname in files:
        safe_download(base + fname, out_dir / fname, description=fname)

    print(f"  Saved to: {out_dir}/")

## Dataset 2 — JavaVFC  (Zenodo 13731781)

In [ ]:
def download_javavfc():
    """
    Downloads the JavaVFC dataset from Zenodo.
    Files: javavfc.jsonl (18 MB), javavfc_extended.jsonl (531 MB)
    NOTE: javavfc_extended.jsonl is large — skipped by default.
          Set INCLUDE_EXTENDED = True in the Configuration cell to include it.
    """
    print("\n[2/4] JavaVFC  (Zenodo record 13731781)")
    out_dir = RAW_DIR / "javavfc"
    out_dir.mkdir(parents=True, exist_ok=True)

    files = {
        "javavfc.jsonl": "https://zenodo.org/records/13731781/files/javavfc.jsonl?download=1",
        "regex.txt": "https://zenodo.org/records/13731781/files/regex.txt?download=1",
    }
    extended_file = {
        "javavfc_extended.jsonl": "https://zenodo.org/records/13731781/files/javavfc_extended.jsonl?download=1",
    }

    for fname, url in files.items():
        safe_download(url, out_dir / fname, description=fname)

    if INCLUDE_EXTENDED:
        for fname, url in extended_file.items():
            print("  [NOTE] Downloading extended dataset (531 MB) — this may take a while...")
            safe_download(url, out_dir / fname, description=fname)
    else:
        print("  [SKIP] javavfc_extended.jsonl (531 MB) — set INCLUDE_EXTENDED=True to download.")

    print(f"  Saved to: {out_dir}/")

## Dataset 3 — CWE-Bench-Java  (iris-sast GitHub)

In [ ]:
def download_cwe_bench_java():
    """
    Downloads CWE-Bench-Java metadata CSVs from the iris-sast/iris GitHub repo.
    NOTE: These files contain CVE/commit references, NOT the actual source code.
          Use the companion script (fetch_cwe_bench_code.py) to pull code
          from GitHub using the commit hashes in project_info.csv + fix_info.csv.
    """
    print("\n[3/4] CWE-Bench-Java  (iris-sast GitHub)")
    out_dir = RAW_DIR / "cwe_bench_java"
    out_dir.mkdir(parents=True, exist_ok=True)

    base = "https://raw.githubusercontent.com/iris-sast/iris/v2/data/"
    files = [
        "project_info.csv",
        "fix_info.csv",
        "fix_info_source_sink.csv",
        "build_info.csv",
        "build_cmds.csv",
    ]

    for fname in files:
        safe_download(base + fname, out_dir / fname, description=fname)

    # Also try to download the HuggingFace version (has more fields)
    hf_base = "https://huggingface.co/datasets/iris-sast/CWE-Bench-Java/resolve/main/"
    hf_files = ["data/train-00000-of-00001.parquet"]
    for fname in hf_files:
        dest = out_dir / Path(fname).name
        safe_download(hf_base + fname, dest, description=Path(fname).name)

    print(f"  Saved to: {out_dir}/")
    print(
        "  NOTE: Run `python 01b_fetch_cwe_bench_code.py` to retrieve"
        " actual Java source code via GitHub API."
    )

## Dataset 4 — CVEFixes  (Kaggle)

In [ ]:
def download_cvefixes_kaggle():
    """
    Downloads the CVEFixes Java dataset from Kaggle.
    Requires kaggle credentials at ~/.kaggle/kaggle.json
      OR env vars: KAGGLE_USERNAME and KAGGLE_KEY

    Dataset: girish17019/cvefixes-vulnerable-and-fixed-code
    Falls back to the original CVEFixes SQLite from Zenodo if Kaggle is unavailable.
    """
    print("\n[4/4] CVEFixes  (Kaggle / Zenodo fallback)")
    out_dir = RAW_DIR / "cvefixes"
    out_dir.mkdir(parents=True, exist_ok=True)

    # ── Try Kaggle API first ─────────────────────────────────────────────────
    kaggle_ok = False
    try:
        import kaggle  # noqa: F401 — just checks import

        kaggle_creds = Path.home() / ".kaggle" / "kaggle.json"
        env_user = os.environ.get("KAGGLE_USERNAME")
        env_key = os.environ.get("KAGGLE_KEY")

        if kaggle_creds.exists() or (env_user and env_key):
            from kaggle.api.kaggle_api_extended import KaggleApiClient
            api = KaggleApiClient()
            api.authenticate()
            print("  Authenticated with Kaggle API.")
            api.dataset_download_files(
                "girish17019/cvefixes-vulnerable-and-fixed-code",
                path=str(out_dir),
                unzip=True,
                quiet=False,
            )
            print(f"  Saved Kaggle dataset to: {out_dir}/")
            kaggle_ok = True
        else:
            print("  [WARN] No Kaggle credentials found.")
    except ImportError:
        print("  [WARN] `kaggle` package not installed. Run: pip install kaggle")
    except Exception as e:
        print(f"  [WARN] Kaggle download failed: {e}")

    # ── Fallback: original CVEFixes SQLite from Zenodo ──────────────────────
    if not kaggle_ok:
        print(
            "  Falling back to original CVEFixes SQLite database from Zenodo."
            "\n  This is the full multi-language DB (~2 GB). "
            "We will filter Java entries during cleaning."
        )
        # CVEFixes v1.0.8 — Zenodo DOI: 10.5281/zenodo.4476563
        # The actual file URL follows the Zenodo record pattern
        zenodo_url = (
            "https://zenodo.org/records/7029359/files/CVEfixes_v1.0.7.zip?download=1"
        )
        dest = out_dir / "CVEfixes.zip"

        print(
            "\n  [ACTION REQUIRED] Zenodo requires a browser-based download for large files."
            "\n  Please manually download CVEFixes from:"
            "\n    https://doi.org/10.5281/zenodo.4476563"
            "\n  and place the .db file at:"
            f"\n    {(out_dir / 'CVEfixes.db').absolute()}"
            "\n\n  Alternatively, set up Kaggle credentials and re-run this script."
        )
        # Write instructions file
        instructions = out_dir / "DOWNLOAD_INSTRUCTIONS.txt"
        instructions.write_text(
            "CVEFixes Dataset Download Instructions\n"
            "======================================\n\n"
            "Option A (Kaggle — recommended for easy Java-only CSV):\n"
            "  1. Create account at https://kaggle.com\n"
            "  2. Go to: https://www.kaggle.com/datasets/girish17019/cvefixes-vulnerable-and-fixed-code\n"
            "  3. Download and extract to this folder.\n\n"
            "Option B (Original Zenodo SQLite — full multi-language DB):\n"
            "  1. Visit: https://doi.org/10.5281/zenodo.4476563\n"
            "  2. Download CVEfixes.db (latest version)\n"
            "  3. Place it in this folder as: CVEfixes.db\n"
            "  The cleaning script will filter for Java entries automatically.\n"
        )
        print(f"  Instructions written to: {instructions}")

## ⚙️ Configuration & Run

In [ ]:
# ── Configuration — edit these variables before running ───────────────────────

# Which datasets to download.
# Use "all" to download everything, or a list subset:
# ["barryallen16", "javavfc", "cwe_bench", "cvefixes"]
DATASETS = "all"

# Set True to also download javavfc_extended.jsonl (~531 MB)
INCLUDE_EXTENDED = False

In [ ]:
# ── Run downloads ──────────────────────────────────────────────────────────────
RAW_DIR.mkdir(parents=True, exist_ok=True)

datasets_list = [DATASETS] if isinstance(DATASETS, str) else DATASETS
run_all = "all" in datasets_list

if run_all or "barryallen16" in datasets_list:
    download_barryallen16()

if run_all or "javavfc" in datasets_list:
    download_javavfc()

if run_all or "cwe_bench" in datasets_list:
    download_cwe_bench_java()

if run_all or "cvefixes" in datasets_list:
    download_cvefixes_kaggle()

print("\n✓ Download phase complete. Raw data in:", RAW_DIR.absolute())
print("  Next step: run the 02_clean_and_convert notebook")